*© Sabrine Bendimerad · [sbendimerad.github.io](https://sbendimerad.github.io) · Licence [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/): you may share and adapt this material with credit, for non-commercial use, under the same licence. Datasets keep their own licences.*

# Noise cancellation 🔇

PCA can also **clean** data. In this exercise, we classify handwritten digits that are hidden under a lot of noise, and use PCA to remove part of it.

## The dataset
`data/noisy_digits.csv` contains 1,797 images of handwritten digits, 8 × 8 pixels each: one row per image, one column per pixel (`pixel_0` to `pixel_63`). They are scikit-learn's digits (pixel values from 0 to 16) to which we added random Gaussian noise with a standard deviation of 4; negative values were then set to 0. `data/noisy_digits_labels.csv` gives the digit of each image (column `digit`).

Because we know how the noise was made, we can also load the **clean** images (`load_digits()`, bundled with scikit-learn) and check how well PCA removes the noise. In real life, you never have the clean version.

1. Import the libraries and load the data (given).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# colour palette of the course
pio.templates["course"] = go.layout.Template(
    layout_colorway=["#3B47C4", "#E4572E", "#2E7D4F", "#F2C14E", "#8E7CC3", "#4BA3C3", "#111111", "#999999"]
)
pio.templates.default = "plotly_white+course"

digits = pd.read_csv("data/noisy_digits.csv")
labels = pd.read_csv("data/noisy_digits_labels.csv")["digit"]
clean = load_digits().data                      # the same images, without noise (same order)
digits.head()

,pixel_0,pixel_1,pixel_2,pixel_3,pixel_4,pixel_5,pixel_6,pixel_7,pixel_8,pixel_9,...,pixel_54,pixel_55,pixel_56,pixel_57,pixel_58,pixel_59,pixel_60,pixel_61,pixel_62,pixel_63
0,4.1,6.6,9.6,9.1,3.4,1.3,3.4,2.0,7.2,3.0,...,5.1,2.8,0.0,0.0,8.4,12.2,7.6,0.0,0.0,0.0
1,0.0,4.6,0.0,15.5,14.7,5.6,0.0,0.0,7.9,0.4,...,0.7,0.7,1.6,0.1,0.0,7.7,17.4,6.4,0.0,0.5
2,0.0,3.6,2.0,2.3,15.5,0.6,0.0,0.0,0.0,0.0,...,2.0,2.4,0.0,0.7,2.8,5.3,6.8,23.7,1.1,0.0
3,0.0,4.8,1.8,10.9,8.4,0.0,0.0,0.7,0.0,1.2,...,9.6,0.5,0.0,0.0,4.6,16.3,13.0,7.1,0.0,7.0
4,5.9,4.4,0.7,0.0,11.8,0.0,2.5,4.1,0.0,5.3,...,0.8,0.0,1.5,0.0,0.0,2.3,14.1,0.0,0.0,0.0


## Looking at the images

2. How many images and pixels are there? What are the smallest and largest pixel values? Display the first 20 noisy images, with their digit as a title.

💡 Hint: `digits.values[i].reshape(8, 8)` turns row `i` into an 8 × 8 image; `plt.subplots(2, 10)` and `ax.imshow(image, cmap="gray_r")` draw them.

3. Split the data into a train set and a test set (25% for the test set, `random_state=0`, `stratify=labels`). Keep the clean version of the test images too: `clean[X_test.index]`.

4. Should we standardise the pixels before the PCA? Think about what the pixels measure, and about the pixels on the border of the images.

## A classifier on the noisy pixels

5. Train a `SVC()` (support vector classifier, default settings) on the noisy training pixels and display its accuracy on the test set. This is our baseline.

## Denoising with PCA

6. Fit a PCA on the training images that keeps **70% of the variance**. How many components does it keep?

7. Denoise the test images: project them on the components (`transform`) and go back to the pixels (`inverse_transform`). Display 10 test images in three rows: noisy, denoised, clean. Then compute the root mean squared difference between the clean images and (a) the noisy images, (b) the denoised images.

💡 Hint: the root mean squared difference between two arrays `a` and `b` is `np.sqrt(((a - b) ** 2).mean())`.

8. Train a new `SVC()` on the PCA features of the training set (the 20 coordinates, `pca.transform(X_train)`), and compute its accuracy on the PCA features of the test set. Compare with question 5.

## Why does it work?

9. Make an image of pure noise with the same shape as the training set: `np.random.default_rng(0).normal(0, 4, X_train.shape)`. Fit a PCA with all the components on it, and another one on the noisy training images. Plot the `explained_variance_` of both (one curve each, log scale for the y axis). What do you notice? Explain why PCA removes noise.

## Choosing the share of variance

10. With 5-fold cross-validation on the **training set** (`StratifiedKFold(5, shuffle=True, random_state=0)`), compare a `SVC()` alone and pipelines `PCA(n_components=share)` → `SVC()` for `share` in 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9 and 0.95. Which share is best? What happens when it is too small or too large?

11. Check your answer to question 4: build a pipeline `StandardScaler()` → `PCA(n_components=0.7)` → `SVC()`, fit it on the training set and compute its test accuracy. How many components does the PCA keep now?